# Chapitre 4 — Retrieval avancé

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-04-retrieval-avance/04_retrieval_avance.ipynb)

Ce notebook contient les 10 exemples du chapitre. OpenAI est facultatif ; BM25, RRF, MMR et le réordonnancement fonctionnent sans clé.

## Ressources utiles

- [OpenAI Responses API](https://developers.openai.com/api/docs/guides/text)
- [Sentence Transformers — Retrieve & Re-Rank](https://sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html)
- [rank-bm25](https://github.com/dorianbrown/rank_bm25)

## 0. Préparer Colab ou Jupyter

Installation de BM25 et Sentence Transformers.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".[retrieval,huggingface]",
    ],
    check=True,
)
print("Environnement du chapitre 4 prêt :", Path.cwd())


## Configuration OpenAI facultative

Activez cette cellule uniquement pour exécuter les exemples 1, 2, 3 et 9.

In [ ]:
# @title Activer les exemples OpenAI 1, 2, 3 et 9
UTILISER_OPENAI = False # @param {type:"boolean"}

if UTILISER_OPENAI:
    import os
    import subprocess
    import sys
    from getpass import getpass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"],
        check=True,
    )
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    print("OpenAI est activé pour ce notebook.")
else:
    print("OpenAI désactivé. Les algorithmes locaux restent exécutables.")


## 1. Query Rewriting

Rendre une question conversationnelle autonome avec OpenAI.

Script correspondant : [`01_query_rewriting.py`](examples/01_query_rewriting.py)

In [ ]:
# ruff: noqa: F811
"""Query Rewriting avec la Responses API d'OpenAI."""

import os
from typing import Any

INSTRUCTIONS = """Tu reformules des questions pour un moteur documentaire.
- Remplace les pronoms et références implicites grâce à l'historique.
- Rends la question autonome.
- Conserve l'intention exacte.
- Ne réponds jamais à la question.
Retourne uniquement la question reformulée."""


def client_openai() -> Any:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
    from openai import OpenAI

    return OpenAI()


def reecrire_requete(
    question: str,
    historique: list[str] | None = None,
    *,
    client: Any = None,
    model: str | None = None,
) -> str:
    """Rend une question autonome en utilisant au plus trois tours récents."""

    derniers = (historique or [])[-3:]
    bloc = "\n".join(f"- {tour}" for tour in derniers) or "Aucun historique."
    response = (client or client_openai()).responses.create(
        model=model or os.getenv("OPENAI_MODEL"),
        instructions=INSTRUCTIONS,
        input=f"Historique :\n{bloc}\n\nQuestion originale : {question}",
    )
    return response.output_text.strip()


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple facultatif : configurez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        tours = ["Nous examinons les conditions de livraison d'une commande en ligne."]
        print(reecrire_requete("Et pour les délais ?", tours))


## 2. Query Expansion et RRF

Diversifier les formulations, puis fusionner les classements.

Script correspondant : [`02_query_expansion.py`](examples/02_query_expansion.py)

In [ ]:
# ruff: noqa: F811
"""Query Expansion avec OpenAI, puis fusion locale par RRF."""

import os
from dataclasses import dataclass
from typing import Any, Protocol


@dataclass(frozen=True)
class Passage:
    page_content: str
    source: str


class Retriever(Protocol):
    def invoke(self, question: str) -> list[Passage]: ...


def generer_variantes(
    question: str,
    n: int = 3,
    *,
    client: Any = None,
    model: str | None = None,
) -> list[str]:
    """Produit plusieurs formulations et conserve la question d'origine."""

    if client is None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        from openai import OpenAI

        client = OpenAI()
    response = client.responses.create(
        model=model or os.getenv("OPENAI_MODEL"),
        instructions="Retourne uniquement les reformulations, une par ligne.",
        input=(
            f"Génère {n} reformulations de cette question avec des synonymes métier, "
            f"un angle plus général et un angle plus précis :\n{question}"
        ),
    )
    variantes = [ligne.strip(" -0123456789.") for ligne in response.output_text.splitlines()]
    return [question, *[variante for variante in variantes if variante]][: n + 1]


def fusion_rrf(listes: list[list[Passage]], constante: int = 60) -> list[Passage]:
    """Fusionne des classements incompatibles à partir de leurs rangs."""

    cumul: dict[tuple[str, str], dict[str, object]] = {}
    for resultats in listes:
        for rang, passage in enumerate(resultats, start=1):
            cle = (passage.source, passage.page_content)
            entree = cumul.setdefault(cle, {"score": 0.0, "passage": passage})
            entree["score"] = float(entree["score"]) + 1.0 / (constante + rang)
    classes = sorted(cumul.values(), key=lambda item: float(item["score"]), reverse=True)
    return [item["passage"] for item in classes]  # type: ignore[misc]


def recherche_elargie(question: str, retriever: Retriever, variantes: list[str], k: int = 5) -> list[Passage]:
    return fusion_rrf([retriever.invoke(variante) for variante in [question, *variantes]])[:k]


if __name__ == "__main__":
    listes = [
        [Passage("Retour sous 30 jours", "retours.md"), Passage("Livraison en 5 jours", "livraison.md")],
        [Passage("Livraison en 5 jours", "livraison.md"), Passage("Retour sous 30 jours", "retours.md")],
    ]
    print("Démonstration RRF :", [passage.source for passage in fusion_rrf(listes)])
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Expansion OpenAI facultative : configurez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        print("Variantes OpenAI :", generer_variantes("Quel est le délai de retour ?"))


## 3. HyDE

Générer un document hypothétique OpenAI avant la recherche.

Script correspondant : [`03_hyde.py`](examples/03_hyde.py)

In [ ]:
# ruff: noqa: F811
"""HyDE : rechercher avec l'embedding d'un document hypothétique OpenAI."""

import math
import os
import re
from collections import Counter
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Passage:
    texte: str
    source: str


def vecteur_mots(texte: str, vocabulaire: list[str]) -> list[float]:
    compte = Counter(re.findall(r"\w+", texte.lower()))
    return [float(compte[mot]) for mot in vocabulaire]


def cosinus(gauche: list[float], droite: list[float]) -> float:
    produit = sum(a * b for a, b in zip(gauche, droite, strict=True))
    norme_g = math.sqrt(sum(valeur**2 for valeur in gauche))
    norme_d = math.sqrt(sum(valeur**2 for valeur in droite))
    return produit / (norme_g * norme_d) if norme_g and norme_d else 0.0


def rediger_hypothese(question: str, *, client: Any = None, model: str | None = None) -> str:
    if client is None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        from openai import OpenAI

        client = OpenAI()
    response = client.responses.create(
        model=model or os.getenv("OPENAI_MODEL"),
        instructions=(
            "Rédige 3 à 5 phrases factuelles comme un manuel officiel. "
            "N'indique jamais qu'il s'agit d'une hypothèse."
        ),
        input=question,
    )
    return response.output_text


def recherche_hyde(question: str, corpus: list[Passage], *, client: Any = None, k: int = 3) -> list[Passage]:
    hypothese = rediger_hypothese(question, client=client)
    vocabulaire = sorted({mot for passage in corpus for mot in re.findall(r"\w+", passage.texte.lower())})
    vecteur_hypothese = vecteur_mots(hypothese, vocabulaire)
    return sorted(
        corpus,
        key=lambda passage: cosinus(vecteur_hypothese, vecteur_mots(passage.texte, vocabulaire)),
        reverse=True,
    )[:k]


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple facultatif : configurez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        documents = [
            Passage("Les retours sont acceptés sous trente jours.", "retours.md"),
            Passage("La livraison standard prend cinq jours.", "livraison.md"),
        ]
        print(recherche_hyde("Quel est le délai de retour ?", documents))


## 4. Recherche hybride

Combiner un signal dense pédagogique et BM25.

Script correspondant : [`04_hybrid_search.py`](examples/04_hybrid_search.py)

In [ ]:
# ruff: noqa: F811
"""Recherche hybride : classement dense, BM25 et fusion RRF."""

import re
from dataclasses import dataclass

from rank_bm25 import BM25Okapi


@dataclass(frozen=True)
class Passage:
    page_content: str
    source: str


def tokeniser(texte: str) -> list[str]:
    return re.findall(r"[a-zà-ÿ0-9\-]+", texte.lower())


def fusion_rrf(listes: list[list[Passage]], constante: int = 60) -> list[Passage]:
    scores: dict[Passage, float] = {}
    for resultats in listes:
        for rang, passage in enumerate(resultats, start=1):
            scores[passage] = scores.get(passage, 0.0) + 1.0 / (constante + rang)
    return sorted(scores, key=scores.get, reverse=True)  # type: ignore[arg-type]


class RechercheHybride:
    def __init__(self, chunks: list[Passage]) -> None:
        self.chunks = chunks
        self.bm25 = BM25Okapi([tokeniser(chunk.page_content) for chunk in chunks])

    def recherche_dense(self, question: str) -> list[Passage]:
        mots = set(tokeniser(question))
        return sorted(
            self.chunks,
            key=lambda chunk: len(mots & set(tokeniser(chunk.page_content))),
            reverse=True,
        )

    def rechercher(self, question: str, k: int = 5) -> list[Passage]:
        dense = self.recherche_dense(question)
        scores = self.bm25.get_scores(tokeniser(question))
        ordre = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
        lexical = [self.chunks[index] for index in ordre]
        return fusion_rrf([dense, lexical])[:k]


if __name__ == "__main__":
    corpus = [
        Passage("Le produit SKU-4892 est garanti deux ans.", "catalogue.md"),
        Passage("La garantie standard couvre vingt-quatre mois.", "garantie.md"),
        Passage("La livraison prend cinq jours.", "livraison.md"),
    ]
    print(RechercheHybride(corpus).rechercher("garantie SKU-4892", k=2))


## 5. Re-ranking

Reclasser les candidats avec un CrossEncoder Hugging Face.

Script correspondant : [`05_reranking.py`](examples/05_reranking.py)

In [ ]:
# ruff: noqa: F811
"""Re-ranking des candidats avec un CrossEncoder Hugging Face."""

from dataclasses import dataclass
from typing import Any, Protocol


@dataclass(frozen=True)
class Passage:
    page_content: str
    source: str


class Retriever(Protocol):
    def invoke(self, question: str) -> list[Passage]: ...


def charger_reranker() -> Any:
    from sentence_transformers import CrossEncoder

    return CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")


def rechercher_et_reclasser(
    question: str,
    retriever: Retriever,
    *,
    reranker: Any = None,
    n_candidats: int = 20,
    n_final: int = 5,
) -> list[Passage]:
    candidats = retriever.invoke(question)[:n_candidats]
    if not candidats:
        return []
    modele = reranker or charger_reranker()
    scores = modele.predict([(question, doc.page_content) for doc in candidats])
    classes = sorted(zip(candidats, scores, strict=True), key=lambda paire: paire[1], reverse=True)
    return [document for document, _ in classes[:n_final]]


class RetrieverExemple:
    def invoke(self, question: str) -> list[Passage]:
        del question
        return [
            Passage("La livraison prend cinq jours.", "livraison.md"),
            Passage("Les retours sont acceptés sous trente jours.", "retours.md"),
        ]


if __name__ == "__main__":
    for passage in rechercher_et_reclasser("Quel est le délai de retour ?", RetrieverExemple()):
        print(passage.source, passage.page_content)


## 6. MMR pas à pas

Équilibrer pertinence et diversité avec une formule explicite.

Script correspondant : [`06_mmr_pseudocode.py`](examples/06_mmr_pseudocode.py)

In [ ]:
# ruff: noqa: F811
"""Implémentation exécutable de Maximal Marginal Relevance (MMR)."""

import math
import re
from collections import Counter
from dataclasses import dataclass


@dataclass(frozen=True)
class Candidat:
    texte: str
    source: str


def similarite(gauche: str, droite: str) -> float:
    a = Counter(re.findall(r"\w+", gauche.lower()))
    b = Counter(re.findall(r"\w+", droite.lower()))
    mots = set(a) | set(b)
    produit = sum(a[mot] * b[mot] for mot in mots)
    norme_a = math.sqrt(sum(valeur**2 for valeur in a.values()))
    norme_b = math.sqrt(sum(valeur**2 for valeur in b.values()))
    return produit / (norme_a * norme_b) if norme_a and norme_b else 0.0


def selection_mmr(
    candidats: list[Candidat],
    question: str,
    k: int,
    lambda_mult: float = 0.6,
) -> list[Candidat]:
    """Équilibre pertinence pour la question et diversité de la sélection."""

    if not 0 <= lambda_mult <= 1:
        raise ValueError("lambda_mult doit être compris entre 0 et 1")
    restants = list(candidats)
    selection: list[Candidat] = []
    while restants and len(selection) < k:
        def score(candidat: Candidat) -> float:
            pertinence = similarite(candidat.texte, question)
            redondance = max(
                (similarite(candidat.texte, retenu.texte) for retenu in selection),
                default=0.0,
            )
            return lambda_mult * pertinence - (1 - lambda_mult) * redondance

        meilleur = max(restants, key=score)
        selection.append(meilleur)
        restants.remove(meilleur)
    return selection


if __name__ == "__main__":
    documents = [
        Candidat("Retour produit possible pendant trente jours.", "retours-1.md"),
        Candidat("Le délai de retour produit est de trente jours.", "retours-2.md"),
        Candidat("Le remboursement arrive sous cinq jours.", "remboursement.md"),
    ]
    print(selection_mmr(documents, "délai de retour et remboursement", 2))


## 7. MMR dans un vector store

Comprendre le contrat d'une recherche MMR intégrée.

Script correspondant : [`07_mmr.py`](examples/07_mmr.py)

In [ ]:
# ruff: noqa: F811
"""Utiliser MMR via l'interface d'un vector store."""

from dataclasses import dataclass


@dataclass(frozen=True)
class Passage:
    page_content: str
    source: str


def selection_diversifiee(
    base_vectorielle,
    question: str,
    k: int = 5,
    fetch_k: int = 25,
    lambda_mult: float = 0.6,
) -> list[Passage]:
    """Délègue au vector store la sélection pertinente et complémentaire."""

    return base_vectorielle.max_marginal_relevance_search(
        query=question,
        k=k,
        fetch_k=fetch_k,
        lambda_mult=lambda_mult,
    )


class VectorStoreExemple:
    """Double pédagogique qui montre le contrat attendu du vector store."""

    def __init__(self, documents: list[Passage]) -> None:
        self.documents = documents

    def max_marginal_relevance_search(
        self,
        *,
        query: str,
        k: int,
        fetch_k: int,
        lambda_mult: float,
    ) -> list[Passage]:
        del query, fetch_k, lambda_mult
        return self.documents[:k]


if __name__ == "__main__":
    store = VectorStoreExemple(
        [
            Passage("Politique de retour", "retours.md"),
            Passage("Délai de remboursement", "remboursement.md"),
        ]
    )
    print(selection_diversifiee(store, "retour et remboursement", k=2))


## 8. Réorganisation en V

Placer les meilleurs passages aux extrémités du prompt.

Script correspondant : [`08_reorder.py`](examples/08_reorder.py)

In [ ]:
# ruff: noqa: F811
"""Réorganisation en V pour limiter l'effet 'lost in the middle'."""

from typing import TypeVar

T = TypeVar("T")


def reordonner_pour_le_prompt(chunks: list[T]) -> list[T]:
    """Place les éléments les mieux classés aux deux extrémités."""

    if len(chunks) <= 2:
        return chunks
    gauche: list[T] = []
    droite: list[T] = []
    for position, chunk in enumerate(chunks):
        (gauche if position % 2 == 0 else droite).append(chunk)
    return gauche + droite[::-1]


if __name__ == "__main__":
    print("Classement initial :", [1, 2, 3, 4, 5, 6])
    print("Ordre dans le prompt :", reordonner_pour_le_prompt([1, 2, 3, 4, 5, 6]))


## 9. Compression contextuelle

Extraire avec OpenAI les phrases utiles de chaque passage.

Script correspondant : [`09_compression.py`](examples/09_compression.py)

In [ ]:
# ruff: noqa: F811
"""Compression contextuelle par extraction avec OpenAI."""

import os
from dataclasses import dataclass
from typing import Any, Protocol


@dataclass(frozen=True)
class Passage:
    page_content: str
    source: str


class Retriever(Protocol):
    def invoke(self, question: str) -> list[Passage]: ...


def compresser_passage(
    question: str,
    passage: Passage,
    *,
    client: Any = None,
    model: str | None = None,
) -> Passage | None:
    if client is None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        from openai import OpenAI

        client = OpenAI()
    response = client.responses.create(
        model=model or os.getenv("OPENAI_MODEL"),
        instructions=(
            "Extrais uniquement les phrases utiles pour répondre à la question. "
            "Si rien n'est utile, réponds exactement HORS_SUJET."
        ),
        input=f"QUESTION\n{question}\n\nPASSAGE\n{passage.page_content}",
    )
    texte = response.output_text.strip()
    return None if texte == "HORS_SUJET" else Passage(texte, passage.source)


def rechercher_et_compresser(
    question: str,
    retriever: Retriever,
    *,
    client: Any = None,
) -> list[Passage]:
    compresses = [
        compresser_passage(question, passage, client=client)
        for passage in retriever.invoke(question)
    ]
    return [passage for passage in compresses if passage is not None]


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple facultatif : configurez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        class RetrieverExemple:
            def invoke(self, question: str) -> list[Passage]:
                del question
                return [
                    Passage(
                        "Les retours sont acceptés sous trente jours. "
                        "Le service client est ouvert le lundi.",
                        "retours.md",
                    )
                ]

        print(rechercher_et_compresser("Quel est le délai de retour ?", RetrieverExemple()))


## 10. Pipeline complet

Assembler les étapes dans un ordre observable et testable.

Script correspondant : [`10_pipeline_ordre.py`](examples/10_pipeline_ordre.py)

In [ ]:
# ruff: noqa: F811
"""Pipeline de retrieval complet : réparer, chercher, fusionner et raffiner."""

from collections.abc import Callable
from dataclasses import dataclass


@dataclass(frozen=True)
class Passage:
    texte: str
    source: str
    score: float


def fusion_rrf(listes: list[list[Passage]], constante: int = 60) -> list[Passage]:
    scores: dict[tuple[str, str], float] = {}
    passages: dict[tuple[str, str], Passage] = {}
    for resultats in listes:
        for rang, passage in enumerate(resultats, start=1):
            cle = (passage.source, passage.texte)
            passages[cle] = passage
            scores[cle] = scores.get(cle, 0.0) + 1.0 / (constante + rang)
    ordre = sorted(scores, key=scores.get, reverse=True)  # type: ignore[arg-type]
    return [passages[cle] for cle in ordre]


def reordonner_en_v(passages: list[Passage]) -> list[Passage]:
    return passages[::2] + passages[1::2][::-1]


def pipeline_retrieval(
    question: str,
    recherches: list[Callable[[str], list[Passage]]],
    *,
    variantes: list[str] | None = None,
    seuil: float = 0.2,
    k: int = 5,
) -> list[Passage]:
    """Orchestre les étapes dans un ordre explicite et testable."""

    requetes = [question, *(variantes or [])]
    listes = [recherche(requete) for requete in requetes for recherche in recherches]
    candidats = fusion_rrf(listes)[:30]
    candidats = sorted(candidats, key=lambda passage: passage.score, reverse=True)
    candidats = [passage for passage in candidats if passage.score >= seuil][:k]
    return reordonner_en_v(candidats)


if __name__ == "__main__":
    corpus = [
        Passage("Retour possible sous trente jours.", "retours.md", 0.95),
        Passage("Remboursement sous cinq jours.", "remboursement.md", 0.80),
        Passage("Livraison standard.", "livraison.md", 0.10),
    ]

    def recherche_locale(question: str) -> list[Passage]:
        del question
        return corpus

    print(pipeline_retrieval("Quel délai de retour ?", [recherche_locale], k=2))


## Bilan

Un retrieval avancé est une cascade : réparer la requête, élargir le rappel, fusionner, reclasser, diversifier, compresser puis ordonner le contexte. Chaque étape doit être évaluée séparément avant d'être conservée.